In [0]:
from pyspark.sql.functions import col

dbutils.widgets.text("batch_id", "full-sample-20261010-01", "Bronze batch ID")
batch_id = dbutils.widgets.get("batch_id").strip()

if not batch_id:
    raise ValueError("batch_id is required")

bronze_batch_df = (
    spark.table("workspace.tmdb_lakehouse.bronze_movie_records")
    .filter(col("batch_id") == batch_id)
)

batch_count = bronze_batch_df.count()
if batch_count == 0:
    raise ValueError(f"No Bronze records found for batch_id={batch_id}")

print(f"Bronze rows in selected batch: {batch_count}")
bronze_batch_df.select(
    "batch_id",
    "load_type",
    "source_extracted_at",
    "window_start",
    "window_end",
).distinct().show(truncate=False)

In [0]:
from pyspark.sql.functions import col, from_json, size
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

movie_schema = StructType([
    StructField("id", LongType(), True),
    StructField("imdb_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("original_title", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("original_language", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("adult", BooleanType(), True),
    StructField("video", BooleanType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("vote_average", DoubleType(), True),
    StructField("vote_count", LongType(), True),
    StructField("budget", LongType(), True),
    StructField("revenue", LongType(), True),

    StructField("genres", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
    ])), True),

    StructField("production_companies", ArrayType(StructType([
        StructField("id", LongType(), True),
        StructField("name", StringType(), True),
        StructField("origin_country", StringType(), True),
    ])), True),

    StructField("credits", StructType([
        StructField("cast", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("credit_id", StringType(), True),
            StructField("name", StringType(), True),
            StructField("character", StringType(), True),
            StructField("order", IntegerType(), True),
        ])), True),
    ]), True),

    StructField("keywords", StructType([
        StructField("keywords", ArrayType(StructType([
            StructField("id", LongType(), True),
            StructField("name", StringType(), True),
        ])), True),
    ]), True),

    StructField("reviews", StructType([
        StructField("results", ArrayType(StructType([
            StructField("id", StringType(), True),
            StructField("author", StringType(), True),
            StructField("author_details", StructType([
                StructField("name", StringType(), True),
                StructField("username", StringType(), True),
                StructField("avatar_path", StringType(), True),
                StructField("rating", DoubleType(), True),
            ]), True),
            StructField("content", StringType(), True),
            StructField("created_at", StringType(), True),
            StructField("updated_at", StringType(), True),
        ])), True),
    ]), True),
])

parsed_movies_df = (
    bronze_batch_df
    .select(
        col("movie_id").alias("bronze_movie_id"),
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        from_json(col("raw_payload"), movie_schema).alias("movie"),
    )
    .select(
        "bronze_movie_id",
        "batch_id",
        "load_type",
        "source_file",
        "source_extracted_at",
        "movie.*",
    )
)

parsed_movies_df.printSchema()

print("Bronze rows in batch:", bronze_batch_df.count())
print("Parsed movie IDs missing:", parsed_movies_df.filter(col("id").isNull()).count())
print("Parsed titles missing:", parsed_movies_df.filter(col("title").isNull()).count())

display(
    parsed_movies_df.select(
        "bronze_movie_id",
        "id",
        "title",
        "budget",
        "revenue",
        size("genres").alias("genre_count"),
        size("production_companies").alias("studio_count"),
        size("credits.cast").alias("cast_count"),
        size("keywords.keywords").alias("keyword_count"),
        size("reviews.results").alias("review_count"),
    ).limit(10)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

# Stop rather than silently accepting records that fail the movie contract.
invalid_movies = parsed_movies_df.filter(
    F.col("id").isNull()
    | F.col("title").isNull()
    | (F.col("id") != F.col("bronze_movie_id"))
)

invalid_count = invalid_movies.count()
if invalid_count:
    display(invalid_movies.select("bronze_movie_id", "id", "title").limit(20))
    raise ValueError(
        f"{invalid_count} records have a missing/mismatched movie ID or missing title. "
        "Do not continue until they are quarantined."
    )

clean_budget = F.when(F.col("budget") > 0, F.col("budget")).otherwise(
    F.lit(None).cast(LongType())
)
clean_revenue = F.when(F.col("revenue") > 0, F.col("revenue")).otherwise(
    F.lit(None).cast(LongType())
)

silver_movies_source_df = (
    parsed_movies_df
    .select(
        F.col("id").alias("movie_id"),
        "imdb_id",
        "title",
        "original_title",
        "overview",
        "original_language",
        F.to_date("release_date", "yyyy-MM-dd").alias("release_date"),
        "runtime",
        "status",
        "adult",
        "video",
        "popularity",
        "vote_average",
        "vote_count",
        clean_budget.alias("budget"),
        clean_revenue.alias("revenue"),
        F.when(
            (clean_budget.isNotNull()) & (clean_revenue.isNotNull()),
            clean_revenue - clean_budget,
        ).cast(LongType()).alias("profit"),
        F.when(
            (clean_budget.isNotNull()) & (clean_revenue.isNotNull()),
            (clean_revenue - clean_budget) / clean_budget,
        ).cast("double").alias("roi"),
        F.when(clean_budget.isNull(), F.lit(None).cast("string"))
        .when(clean_budget < 10000000, F.lit("low"))
        .when(clean_budget < 100000000, F.lit("medium"))
        .otherwise(F.lit("high"))
        .alias("budget_tier"),
        F.col("batch_id").alias("source_batch_id"),
        "source_file",
        "source_extracted_at",
    )
)

business_columns = [
    "movie_id",
    "imdb_id",
    "title",
    "original_title",
    "overview",
    "original_language",
    "release_date",
    "runtime",
    "status",
    "adult",
    "video",
    "popularity",
    "vote_average",
    "vote_count",
    "budget",
    "revenue",
    "profit",
    "roi",
    "budget_tier",
]

silver_movies_source_df = (
    silver_movies_source_df
    .withColumn(
        "row_hash",
        F.sha2(
            F.to_json(F.struct(*[F.col(name) for name in business_columns])),
            256,
        ),
    )
    .withColumn("load_timestamp", F.current_timestamp())
)

print("Silver movie rows prepared:", silver_movies_source_df.count())
silver_movies_source_df.printSchema()

display(
    silver_movies_source_df.select(
        "movie_id",
        "title",
        "release_date",
        "budget",
        "revenue",
        "profit",
        "roi",
        "budget_tier",
        "source_batch_id",
        "source_extracted_at",
    ).limit(10)
)

In [0]:
print(
    "Unknown-budget rows with a tier:",
    silver_movies_source_df.filter(
        F.col("budget").isNull() & F.col("budget_tier").isNotNull()
    ).count()
)

In [0]:
from pyspark.sql import functions as F

duplicate_movie_keys = (
    silver_movies_source_df
    .groupBy("movie_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_financials = silver_movies_source_df.filter(
    (
        F.col("budget").isNotNull()
        & F.col("revenue").isNotNull()
        & (
            (F.col("profit") != F.col("revenue") - F.col("budget"))
            | (
                F.abs(
                    F.col("roi")
                    - ((F.col("revenue") - F.col("budget")) / F.col("budget"))
                ) > 1e-9
            )
        )
    )
    | (
        (F.col("budget").isNull() | F.col("revenue").isNull())
        & (F.col("profit").isNotNull() | F.col("roi").isNotNull())
    )
).count()

invalid_budget_tiers = silver_movies_source_df.filter(
    (F.col("budget").isNull() & F.col("budget_tier").isNotNull())
    | (
        (F.col("budget") > 0)
        & (F.col("budget") < 10000000)
        & (F.col("budget_tier") != "low")
    )
    | (
        (F.col("budget") >= 10000000)
        & (F.col("budget") < 100000000)
        & (F.col("budget_tier") != "medium")
    )
    | (
        (F.col("budget") >= 100000000)
        & (F.col("budget_tier") != "high")
    )
).count()

missing_hashes = silver_movies_source_df.filter(
    F.col("row_hash").isNull()
).count()

print("Rows prepared:", silver_movies_source_df.count())
print("Duplicate movie keys:", duplicate_movie_keys)
print("Invalid financial calculations:", invalid_financials)
print("Invalid budget tiers:", invalid_budget_tiers)
print("Missing row hashes:", missing_hashes)

if any([duplicate_movie_keys, invalid_financials, invalid_budget_tiers, missing_hashes]):
    raise ValueError("Silver movie validation failed; do not write the table.")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb_lakehouse.silver_movies_cleansed (
    movie_id BIGINT,
    imdb_id STRING,
    title STRING,
    original_title STRING,
    overview STRING,
    original_language STRING,
    release_date DATE,
    runtime INT,
    status STRING,
    adult BOOLEAN,
    video BOOLEAN,
    popularity DOUBLE,
    vote_average DOUBLE,
    vote_count BIGINT,
    budget BIGINT,
    revenue BIGINT,
    profit BIGINT,
    roi DOUBLE,
    budget_tier STRING,
    source_batch_id STRING,
    source_file STRING,
    source_extracted_at TIMESTAMP,
    row_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("DESCRIBE TABLE workspace.tmdb_lakehouse.silver_movies_cleansed").show(truncate=False)